# ApprovalMax Native Databricks Dashboard Refresh

Creates dashboard-ready Delta tables for Databricks SQL / AI-BI Dashboards.

Dashboard layer:

`workspace.engineer_support_monitoring`

Output tables:

- `dashboard_layer_row_counts`
- `dashboard_pipeline_run_summary`
- `dashboard_etl_step_timeline`
- `dashboard_quality_status`
- `dashboard_gold_documents`
- `dashboard_kpi_snapshot`


In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, DoubleType

catalog = 'workspace'
bronze_schema = 'engineer_support_bronze'
silver_schema = 'engineer_support_silver'
vault_schema = 'engineer_support_vault'
gold_schema = 'engineer_support_gold'
monitoring_schema = 'engineer_support_monitoring'

spark.sql(f'USE CATALOG {catalog}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{monitoring_schema}')

def full_table(schema, table):
    return f'{catalog}.{schema}.{table}'

def table_exists(full_name: str) -> bool:
    try:
        spark.table(full_name).limit(1).count()
        return True
    except Exception:
        return False

def safe_count(full_name: str):
    try:
        return spark.table(full_name).count()
    except Exception:
        return None

refreshed_at = datetime.now(timezone.utc)
print(f'Refreshing dashboard tables at {refreshed_at}')


In [ ]:
# 1) Layer row counts
tables_to_count = [
    ('bronze', full_table(bronze_schema, 'approvalmax_cdc_raw')),
    ('silver', full_table(silver_schema, 'companies_current')),
    ('silver', full_table(silver_schema, 'finance_documents_current')),
    ('silver', full_table(silver_schema, 'approval_events')),
    ('vault', full_table(vault_schema, 'hub_company')),
    ('vault', full_table(vault_schema, 'hub_finance_document')),
    ('vault', full_table(vault_schema, 'hub_approval_event')),
    ('vault', full_table(vault_schema, 'link_document_company')),
    ('vault', full_table(vault_schema, 'link_document_approval_event')),
    ('vault', full_table(vault_schema, 'sat_finance_document_status')),
    ('vault', full_table(vault_schema, 'sat_finance_document_amount')),
    ('vault', full_table(vault_schema, 'sat_approval_event_detail')),
    ('gold', full_table(gold_schema, 'fact_approval_document_lifecycle')),
    ('gold', full_table(gold_schema, 'fact_approval_document_lifecycle_dbt')),
    ('monitoring', full_table(monitoring_schema, 'etl_audit_log')),
    ('monitoring', full_table(monitoring_schema, 'great_expectations_results')),
]

row_count_schema = StructType([
    StructField('layer', StringType(), False),
    StructField('table_name', StringType(), False),
    StructField('row_count', LongType(), True),
    StructField('exists_flag', LongType(), False),
    StructField('refreshed_at', TimestampType(), False),
])

rows = []
for layer, table_name in tables_to_count:
    cnt = safe_count(table_name)
    rows.append({
        'layer': layer,
        'table_name': table_name,
        'row_count': int(cnt) if cnt is not None else None,
        'exists_flag': 1 if cnt is not None else 0,
        'refreshed_at': refreshed_at,
    })

spark.createDataFrame(rows, schema=row_count_schema) \
    .write.format('delta').mode('overwrite').option('overwriteSchema', 'true') \
    .saveAsTable(full_table(monitoring_schema, 'dashboard_layer_row_counts'))

display(spark.table(full_table(monitoring_schema, 'dashboard_layer_row_counts')))


In [ ]:
# 2) ETL audit dashboard tables
audit_table = full_table(monitoring_schema, 'etl_audit_log')

if table_exists(audit_table):
    audit = spark.table(audit_table)
    audit.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(full_table(monitoring_schema, 'dashboard_etl_step_timeline'))

    summary = (
        audit.groupBy('run_id')
        .agg(
            F.min('started_at').alias('run_started_at'),
            F.max('ended_at').alias('run_ended_at'),
            F.sum('duration_seconds').alias('total_duration_seconds'),
            F.count('*').alias('step_count'),
            F.sum(F.when(F.col('status') == 'FAILED', 1).otherwise(0)).alias('failed_step_count'),
            F.sum(F.when(F.col('status') == 'SUCCESS', 1).otherwise(0)).alias('success_step_count'),
        )
        .withColumn('run_status', F.when(F.col('failed_step_count') > 0, F.lit('FAILED')).otherwise(F.lit('SUCCESS')))
        .withColumn('refreshed_at', F.lit(refreshed_at))
    )
    summary.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(full_table(monitoring_schema, 'dashboard_pipeline_run_summary'))
else:
    print(f'Missing audit table: {audit_table}')


In [ ]:
# 3) Great Expectations status table
gx_table = full_table(monitoring_schema, 'great_expectations_results')

if table_exists(gx_table):
    gx = spark.table(gx_table).withColumn('refreshed_at', F.lit(refreshed_at))
    gx.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(full_table(monitoring_schema, 'dashboard_quality_status'))
else:
    print(f'Missing GE results table: {gx_table}')


In [ ]:
# 4) Gold document dashboard table. Prefer dbt Gold model when available.
dbt_gold = full_table(gold_schema, 'fact_approval_document_lifecycle_dbt')
notebook_gold = full_table(gold_schema, 'fact_approval_document_lifecycle')
gold_source = dbt_gold if table_exists(dbt_gold) else notebook_gold

if table_exists(gold_source):
    gold = spark.table(gold_source).withColumn('refreshed_at', F.lit(refreshed_at))
    gold.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(full_table(monitoring_schema, 'dashboard_gold_documents'))
else:
    print('No Gold lifecycle table found yet')


In [ ]:
# 5) KPI snapshot table
kpi_schema = StructType([
    StructField('metric_name', StringType(), False),
    StructField('metric_value', DoubleType(), True),
    StructField('refreshed_at', TimestampType(), False),
])

kpis = []
gold_dashboard = full_table(monitoring_schema, 'dashboard_gold_documents')
row_counts = full_table(monitoring_schema, 'dashboard_layer_row_counts')
quality = full_table(monitoring_schema, 'dashboard_quality_status')

if table_exists(gold_dashboard):
    g = spark.table(gold_dashboard)
    kpis.append({'metric_name': 'gold_document_count', 'metric_value': float(g.count()), 'refreshed_at': refreshed_at})
    kpis.append({'metric_name': 'total_amount', 'metric_value': float(g.select(F.coalesce(F.sum('total_amount'), F.lit(0))).collect()[0][0]), 'refreshed_at': refreshed_at})
    kpis.append({'metric_name': 'sla_breach_count', 'metric_value': float(g.filter(F.col('approval_sla_breach_flag') == 1).count()), 'refreshed_at': refreshed_at})
    kpis.append({'metric_name': 'avg_approval_cycle_minutes', 'metric_value': float(g.select(F.coalesce(F.avg('approval_cycle_time_minutes'), F.lit(0))).collect()[0][0]), 'refreshed_at': refreshed_at})

if table_exists(quality):
    q = spark.table(quality)
    kpis.append({'metric_name': 'failed_expectation_count', 'metric_value': float(q.filter(F.col('success') == False).count()), 'refreshed_at': refreshed_at})

if table_exists(row_counts):
    rc = spark.table(row_counts)
    kpis.append({'metric_name': 'available_table_count', 'metric_value': float(rc.filter(F.col('exists_flag') == 1).count()), 'refreshed_at': refreshed_at})

spark.createDataFrame(kpis, schema=kpi_schema).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(full_table(monitoring_schema, 'dashboard_kpi_snapshot'))
display(spark.table(full_table(monitoring_schema, 'dashboard_kpi_snapshot')))
print('Dashboard refresh complete')
